In [47]:
import pandas as pd
from wittgenstein import RIPPER
import json
from sklearn.metrics import accuracy_score

This demonstration combines LLM feature extraction with rule inference to produce both flexible predictions and explicit, interpretable rules. Results from this small-scale evaluation suggest this hybrid approach may improve classification performance while preserving the interpretability of rule-based models.

Accuracy / F1:
- LLM judge (zero-shot): 0.56 / 0.23
- LLM judge (10-shot): 0.69 / 0.58
- RIPPER on structured features only: 0.61 / 0.50
- RIPPER on structured + LLM-derived features (zero-shot): 0.79 / 0.75


In [2]:
# Load the data
# Full dataset can be found here: https://www.consumerfinance.gov/data-research/consumer-complaints/#get-the-data
# Note: the csv is ~8.5GB

full_df = pd.read_csv('datasets/cfpb_complaints.csv')

In [3]:
full_df.shape

(14533627, 18)

In [4]:
features = ['Issue', 'Sub-issue']
text_feat = 'Consumer complaint narrative'
pre_class_feat = 'Company response to consumer'
class_feat = 'monetary_relief'

In [5]:
# Data cleaning: filter to just credit card debt

df = full_df[full_df['Sub-product']=='Credit card debt']
df = df.drop('Sub-product', axis=1)

In [6]:
# Dropn null values

df = df.dropna(subset=features + [text_feat] + [pre_class_feat])
df.shape

(78128, 17)

In [7]:
df['Company response to consumer'].value_counts()

Company response to consumer
Closed with explanation            60102
Closed with non-monetary relief    16645
Closed with monetary relief         1021
Untimely response                    358
In progress                            2
Name: count, dtype: int64

In [8]:
df = df.copy()
del(full_df)

In [9]:
# Add class feature: positive if outcome is closed with monetary relief, otherwise negative

df[class_feat] = df[pre_class_feat].map(lambda x:x=='Closed with monetary relief')
df.head(1)

,Date received,Product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,monetary_relief
56,2023-12-28,Debt collection,False statements or representation,Attempted to collect wrong amount,"XXXX 2021, amount of XXXX to collect is Not ac...",NaN,Resurgent Capital Services L.P.,KY,402XX,NaN,Consent provided,Web,2023-12-28,Closed with explanation,Yes,NaN,8070648,False


In [10]:
df = df[features + [text_feat, class_feat]]
df

,Issue,Sub-issue,Consumer complaint narrative,monetary_relief
56,False statements or representation,Attempted to collect wrong amount,"XXXX 2021, amount of XXXX to collect is Not ac...",False
91,Attempts to collect debt not owed,Debt was paid,Around 2012 I paid this card with proof from b...,False
168,Attempts to collect debt not owed,Debt was result of identity theft,This particular account situation that is late...,False
228,Written notification about debt,Didn't receive notice of right to dispute,I am being sued by XXXX XXXX XXXX. \nI have ca...,False
487,Attempts to collect debt not owed,Debt was paid,Just want to express that I am very concern an...,False
...,...,...,...,...
14532837,Attempts to collect debt not owed,Debt was paid,RE : XXXX XXXX ; CFPB Case # : XXXX Account Nu...,False
14533089,Attempts to collect debt not owed,Debt is not yours,To Whom It May Concern : I noticed that you ar...,False
14533495,Communication tactics,"You told them to stop contacting you, but they...",I frequently get calls from a place called XXX...,False
14533498,Electronic communications,Frequent or repeated messages,Account was charged off and on my credit after...,False


In [11]:
# Take total sample of 700

sub_df = df.groupby(class_feat, group_keys=False).apply(lambda x: x.sample(350, random_state=42))
print(sub_df.shape)
sub_df.head(1)

(700, 4)


/var/folders/9y/p6kdqql91ln01c2snvs9c5740000gn/T/ipykernel_65322/552435020.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub_df = df.groupby(class_feat, group_keys=False).apply(lambda x: x.sample(350, random_state=42))


,Issue,Sub-issue,Consumer complaint narrative,monetary_relief
4628787,Communication tactics,"You told them to stop contacting you, but they...","They call, text and email me several times per...",False


In [12]:
# Split the data into train/test sets (400/300)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    sub_df.drop([class_feat], axis=1), sub_df[class_feat], random_state=42, stratify=sub_df[class_feat], test_size=300)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((400, 3), (300, 3), (400,), (300,))

In [13]:
X_train.head(1)

,Issue,Sub-issue,Consumer complaint narrative
10024603,False statements or representation,Attempted to collect wrong amount,I had a walmart cred it card through synchron...


In [14]:
end

NameError: name 'end' is not defined

In [38]:
# First test is RIPPER alone, on just categorical Issue and Sub-issue features (no text features)

rip = RIPPER(random_state=42)
rip.fit(X_train.drop(text_feat, axis=1), y_train, class_feat=class_feat)
rip.out_model()

[[Sub-issue=Debtwaspaid] V
[Issue=Falsestatementsorrepresentation]]


In [41]:
rip.score(X_test.drop(text_feat, axis=1), y_test), rip.score(X_test.drop(text_feat, axis=1), y_test, score_function=f1_score)

(0.6133333333333333, np.float64(0.5042735042735043))

In [18]:
# Save training set

train_text_classfeat_subset = pd.concat([X_train[text_feat], y_train], axis=1)
train_text_classfeat_subset = train_text_classfeat_subset[[text_feat, class_feat]]
print(train_text_classfeat_subset.shape)
train_text_classfeat_subset.to_csv('datasets/cfpb_sample_train.csv')
train_text_classfeat_subset.head(1)

(400, 2)


,Consumer complaint narrative,monetary_relief
10024603,I had a walmart cred it card through synchron...,True


In [19]:
# Define text features and prompt

text_features = {
    "describes_specific_transaction": {
        "description": "Does the consumer describe a specific financial transaction with concrete details (dates, amounts, payment method, confirmation numbers)?",
        "values": ["yes", "no"]
    },
    "payment_made_not_credited": {
        "description": "Does the consumer claim they made a payment that was not properly applied or acknowledged by the company?",
        "values": ["yes", "no"]
    },
    "disputes_specific_fee": {
        "description": "Does the consumer dispute a specific fee such as a late fee, interest charge, or cancellation fee as unfair or incorrect?",
        "values": ["yes", "no"]
    },
    "product_or_service_not_received": {
        "description": "Does the consumer claim they paid for a product or service that was never delivered, was defective, or was returned but not refunded?",
        "values": ["yes", "no"]
    },
    "company_broke_agreement": {
        "description": "Does the consumer claim the company failed to honor a prior agreement, promise, or arrangement (e.g., promised to waive fees, agreed to a payment plan, said account would be closed)?",
        "values": ["yes", "no"]
    },
    "prior_attempts_to_resolve": {
        "description": "How many prior attempts has the consumer made to resolve this issue directly with the company?",
        "values": ["none_mentioned", "one", "multiple"]
    },
    "has_documentation": {
        "description": "Does the consumer mention having or attaching supporting evidence such as receipts, bank statements, confirmation emails, or tracking numbers?",
        "values": ["yes", "no"]
    },
    "cites_legal_statutes": {
        "description": "Does the consumer cite specific laws, statutes, or USC sections (e.g., FCRA, FDCPA, UCC, Fair Credit Act)?",
        "values": ["yes", "no"]
    },
    "requests_refund_or_reversal": {
        "description": "Does the consumer explicitly request money back, a refund, a credit, or reversal of charges?",
        "values": ["yes", "no"]
    },
    "complaint_narrative_style": {
        "description": "What best describes the overall writing style of the complaint?",
        "values": ["personal_story", "formal_legal_letter", "brief_statement", "template_dispute"]
    },
}
FEATURIZATION_PROMPT = \
"""You are a feature extractor for consumer financial complaints about debt collection.
Read the complaint and return a JSON object with ONLY these keys and ONLY the allowed values.

{feature_schema}

Return ONLY valid JSON. No explanation or commentary.

COMPLAINT:
{narrative}""".replace('{feature_schema}', str(text_features))

In [20]:
FEATURIZATION_PROMPT

"You are a feature extractor for consumer financial complaints about debt collection.\nRead the complaint and return a JSON object with ONLY these keys and ONLY the allowed values.\n\n{'describes_specific_transaction': {'description': 'Does the consumer describe a specific financial transaction with concrete details (dates, amounts, payment method, confirmation numbers)?', 'values': ['yes', 'no']}, 'payment_made_not_credited': {'description': 'Does the consumer claim they made a payment that was not properly applied or acknowledged by the company?', 'values': ['yes', 'no']}, 'disputes_specific_fee': {'description': 'Does the consumer dispute a specific fee such as a late fee, interest charge, or cancellation fee as unfair or incorrect?', 'values': ['yes', 'no']}, 'product_or_service_not_received': {'description': 'Does the consumer claim they paid for a product or service that was never delivered, was defective, or was returned but not refunded?', 'values': ['yes', 'no']}, 'company_bro

In [21]:
# Utility code for parsing json outputs

def parse_llm_json(raw):
    """Extract a JSON dict from an LLM response that may contain
    markdown fences, newlines, or preamble text."""
    cleaned = raw.strip()
    # strip markdown code fences
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[-1]  # remove opening ```json
        cleaned = cleaned.rsplit("```", 1)[0]  # remove closing ```
    # try parsing directly
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        # last resort: find first { to last }
        start = cleaned.find("{")
        end = cleaned.rfind("}") + 1
        if start != -1 and end > start:
            return json.loads(cleaned[start:end])
        return {"_error": "parse_failed", "_raw": raw}

In [22]:
# Code for making asyncronous calls

import asyncio
import json
import pandas as pd
from anthropic import AsyncAnthropic
from tqdm.asyncio import tqdm_asyncio

model = 'claude-sonnet-4-6'
SEMAPHORE = asyncio.Semaphore(10)
client = AsyncAnthropic()

async def generate_example_text(model, prompt):
    async with SEMAPHORE:
        messages = [{'role': 'user', 'content': prompt}]        
        response = await client.messages.create(
            model=model,
            temperature=0,
            messages=messages,
            max_tokens=256
        )
        return response.content[0].text

async def featurize_X_texts(X, text_feat):
    narratives = X[text_feat]
    print(narratives[:5])
    prompts = [FEATURIZATION_PROMPT.replace('{narrative}', narrative) for narrative in narratives]
    tasks = [generate_example_text(model, prompt) for prompt in prompts]
    results = await tqdm_asyncio.gather(*tasks, desc='Featurizing texts')
    features = [parse_llm_json(result) for result in results]
    return features

In [23]:
# Use an LLM to featurize the text

featurized_texts_X_train = await featurize_X_texts(X_train, text_feat)

10024603    I had  a walmart cred it card through synchron...
8295160     I am submitting this complaint against Midland...
7209917     I demand a cease and desist of all illegal act...
12946958    I attempted to call my creditor to resolve my ...
9600151     Navy Federal Credit Union ( NFCU ) still has n...
Name: Consumer complaint narrative, dtype: object


Featurizing texts: 100%|██████████████████████████████████████████| 400/400 [04:46<00:00,  1.39it/s]


In [24]:
json.dump(featurized_texts_X_train, open('datasets/cfpb_featurized_texts_train.json', 'w+'))

In [25]:
featurized_X_train = pd.concat([X_train.reset_index(drop=True), pd.DataFrame(featurized_texts_X_train)], axis=1)
featurized_X_train.shape

(400, 13)

In [26]:
featurized_X_train.head()

,Issue,Sub-issue,Consumer complaint narrative,describes_specific_transaction,payment_made_not_credited,disputes_specific_fee,product_or_service_not_received,company_broke_agreement,prior_attempts_to_resolve,has_documentation,cites_legal_statutes,requests_refund_or_reversal,complaint_narrative_style
0,False statements or representation,Attempted to collect wrong amount,I had a walmart cred it card through synchron...,yes,yes,yes,no,yes,multiple,no,no,no,personal_story
1,Took or threatened to take negative or legal a...,Threatened or suggested your credit would be d...,I am submitting this complaint against Midland...,no,no,no,no,no,one,no,yes,no,formal_legal_letter
2,Attempts to collect debt not owed,Debt is not yours,I demand a cease and desist of all illegal act...,no,no,no,no,yes,none_mentioned,yes,yes,no,formal_legal_letter
3,Took or threatened to take negative or legal a...,Sued you without properly notifying you of law...,I attempted to call my creditor to resolve my ...,no,no,no,no,no,one,no,no,no,personal_story
4,Attempts to collect debt not owed,Debt is not yours,Navy Federal Credit Union ( NFCU ) still has n...,yes,yes,yes,no,yes,multiple,yes,no,yes,personal_story


In [42]:
# Use wittgenstein to train a model on categorical + featurized data

rip = RIPPER(random_state=42)
rip.fit(featurized_X_train.drop(text_feat, axis=1), y_train)

In [43]:
rip.out_model()

[[describes_specific_transaction=yes ^ disputes_specific_fee=yes ^ Issue=Attemptstocollectdebtnotowed] V
[complaint_narrative_style=personal_story ^ describes_specific_transaction=yes ^ requests_refund_or_reversal=yes ^ Sub-issue=Attemptedtocollectwrongamount ^ disputes_specific_fee=yes] V
[complaint_narrative_style=personal_story ^ describes_specific_transaction=yes ^ Issue=Communicationtactics] V
[describes_specific_transaction=yes ^ Sub-issue=Debtisnotyours] V
[complaint_narrative_style=personal_story ^ disputes_specific_fee=yes] V
[complaint_narrative_style=personal_story ^ product_or_service_not_received=yes] V
[disputes_specific_fee=yes] V
[Issue=Attemptstocollectdebtnotowed ^ describes_specific_transaction=yes ^ Sub-issue=Debtwasresultofidentitytheft ^ prior_attempts_to_resolve=multiple]]


In [29]:
# Featurize the test set

featurized_texts_X_test = await featurize_X_texts(X_test, text_feat)

13135039    Navy Federal Credit Union took my entire Feder...
1839010     I received a letter from Hunt & Khan, P.A. loc...
8665513     To whom it may concern, Back in late 2015 I al...
8413071     I purchesed technical support membership for {...
2948326     I have been ordering from this company 's cata...
Name: Consumer complaint narrative, dtype: object


Featurizing texts: 100%|██████████████████████████████████████████| 300/300 [01:17<00:00,  3.88it/s]


In [30]:
json.dump(featurized_texts_X_test, open('datasets/cfpb_featurized_texts_test.json', 'w+'))

In [31]:
featurized_X_test = pd.concat([X_test.reset_index(drop=True), pd.DataFrame(featurized_texts_X_test)], axis=1)
featurized_X_test.shape

(300, 13)

In [45]:
# Score the model with featurized data

rip.score(featurized_X_test.drop(text_feat, axis=1), y_test), rip.score(featurized_X_test.drop(text_feat, axis=1), y_test, score_function=f1_score)

(0.7866666666666666, np.float64(0.7480314960629921))

In [33]:
# Next, we'll test a zero-shot judge

JUDGE_PROMPT = """Based on this consumer debt collection complaint, will the consumer receive monetary relief (e.g., refund, fee reversal, credit, or payment correction)?

Issue: {issue}
Sub-issue: {sub_issue}

Complaint narrative:
{narrative}

Answer with ONLY a JSON object: {{"monetary_relief": true}} or {{"monetary_relief": false}}"""

In [34]:
async def judge_X_texts(X, issue_feat, sub_issue_feat, narrative_feat):
    prompts = []
    for issue, sub_issue, narrative in zip(X[issue_feat], X[sub_issue_feat], X[narrative_feat]):
        prompt = JUDGE_PROMPT
        prompt = prompt.replace('{issue}', issue)
        prompt = prompt.replace('{sub_issue}', sub_issue)
        prompt = prompt.replace('{narrative}', narrative)
        prompts.append(prompt)
    tasks = [generate_example_text(model, prompt) for prompt in prompts]
    results = await tqdm_asyncio.gather(*tasks, desc='Judging texts')
    features = [parse_llm_json(result) for result in results]
    return features

In [35]:
judged_texts_X_test = await judge_X_texts(X_test, 'Issue', 'Sub-issue', text_feat)

Judging texts: 100%|██████████████████████████████████████████████| 300/300 [02:22<00:00,  2.11it/s]


In [36]:
json.dump(judged_texts_X_test, open('datasets/cfpb_judged_texts_test.json', 'w+'))

In [37]:
judge_preds = [example['monetary_relief'] for example in judged_texts_X_test]
judge_preds[:5]

[False, False, False, True, False]

In [48]:
accuracy_score(y_test, judge_preds), f1_score(y_test, judge_preds)

(0.5633333333333334, np.float64(0.23391812865497075))

In [49]:
# The last test is a few-shot judge

Xy_train = pd.concat([X_train, y_train], axis=1).sort_values('monetary_relief', ascending=False)
Xy_train

,Issue,Sub-issue,Consumer complaint narrative,monetary_relief
10024603,False statements or representation,Attempted to collect wrong amount,I had a walmart cred it card through synchron...,True
10846741,False statements or representation,Attempted to collect wrong amount,"Hello, I purchased three items of jewelry from...",True
7536604,False statements or representation,Attempted to collect wrong amount,Bank of America is reporting my credit card ac...,True
14416854,Attempts to collect debt not owed,Debt was result of identity theft,XXXX money Lion accounts are reporting on XXXX...,True
11653040,Took or threatened to take negative or legal a...,Seized or attempted to seize your property,A credit card recovery firm obtained a Levy at...,True
...,...,...,...,...
14109993,Took or threatened to take negative or legal a...,Collected or attempted to collect exempt funds,"XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX, XXXX....",False
10000439,Attempts to collect debt not owed,Debt was result of identity theft,"Please remove all accounts listed, my informat...",False
3112581,Attempts to collect debt not owed,Debt was paid,Direct Recovery Services LLC took {$210.00} fr...,False
6479517,Threatened to contact someone or share informa...,Talked to a third-party about your debt,My cousin received a text message tonight from...,False


In [50]:
SHOT_SUBPROMPT = \
"""Issue: {issue}
Sub-issue: {sub_issue}
Complaint narrative:
{narrative}
"""
n = 5

pos_examples = list(zip(Xy_train.head(n)['Issue'], Xy_train.head(n)['Sub-issue'], Xy_train.head(n)[text_feat]))
neg_examples = list(zip(Xy_train.tail(n)['Issue'], Xy_train.tail(n)['Sub-issue'], Xy_train.tail(n)[text_feat]))

shots = []
for i, ((p_issue, p_sub, p_narr), (n_issue, n_sub, n_narr)) in enumerate(zip(pos_examples, neg_examples)):
    for label, issue, sub_issue, narrative in [("true", p_issue, p_sub, p_narr), ("false", n_issue, n_sub, n_narr)]:
        shot = f'Example {len(shots)+1}:\n'
        shot = shot + SHOT_SUBPROMPT
        shot = shot.replace('{issue}', issue)
        shot = shot.replace('{sub_issue}', sub_issue)
        shot = shot.replace('{narrative}', narrative)
        shot = shot + f'-> {{"monetary_relief": {label}}}'
        shots.append(shot)

shots = '\n\n'.join(shots)
shots

'Example 1:\nIssue: False statements or representation\nSub-issue: Attempted to collect wrong amount\nComplaint narrative:\nI had  a walmart cred it card through synchrony bank. I was behind on payments and they sent it to  XXXX   XXXX   XXXX   XXXX  for collection. I called them on  XXXX     XXXX    2016  and made payment arrangements to pay off the balance of  XXXX  I arranged post dated payments which were made by direct debit of my checking account. I was given the impression that this would pay off the balance and that would be the end of it. I received my next statement and the balance was almost the same and I saw a late charge had been added which was more than my payment. I called   XXXX   XXXX   back for an explanation. they explained that the charges were added into the 6 month payment plan. I finished the payment plan and now  Walmart is  calling me for the remaining balance of  XXXX . They show no record of payment arrangements.No one can give me a copy of what I was told 

In [51]:
FEWSHOT_JUDGE_PROMPT = """Based on this consumer debt collection complaint, will the consumer receive monetary relief (e.g., refund, fee reversal, credit, or payment correction)?

Examples:
{shots_subprompt}

--------------

Issue: {issue}
Sub-issue: {sub_issue}

Complaint narrative:
{narrative}

Answer with ONLY a JSON object: {{"monetary_relief": true}} or {{"monetary_relief": false}}"""


In [52]:
async def fewshot_judge_X_texts(X, issue_feat, sub_issue_feat, narrative_feat, shots):
    prompts = []
    for issue, sub_issue, narrative in zip(X[issue_feat], X[sub_issue_feat], X[narrative_feat]):
        prompt = FEWSHOT_JUDGE_PROMPT
        prompt = prompt.replace('{shots_subprompt}', shots)
        prompt = prompt.replace('{issue}', issue)
        prompt = prompt.replace('{sub_issue}', sub_issue)
        prompt = prompt.replace('{narrative}', narrative)
        prompts.append(prompt)
    tasks = [generate_example_text(model, prompt) for prompt in prompts]
    results = await tqdm_asyncio.gather(*tasks, desc='Fewshot judging texts')
    features = [parse_llm_json(result) for result in results]
    return features

In [53]:
fewshot_judged_texts_X_test = await fewshot_judge_X_texts(X_test, 'Issue', 'Sub-issue', text_feat, shots)

Fewshot judging texts: 100%|██████████████████████████████████████| 300/300 [00:44<00:00,  6.68it/s]


In [54]:
json.dump(fewshot_judged_texts_X_test, open('datasets/cfpb_fewshow_judged_texts_test.json', 'w+'))

In [55]:
fewshot_judged_preds = [example['monetary_relief'] for example in fewshot_judged_texts_X_test]
fewshot_judged_preds[:5]

[True, False, False, True, True]

In [57]:
accuracy_score(y_test, fewshot_judged_preds), f1_score(y_test, fewshot_judged_preds)

(0.6933333333333334, np.float64(0.5818181818181818))